# OPT3 / FD3 / FD5 Benchmarks Against A Fine FD5 Reference

This notebook compares a well-sampled `FD5` reference against coarse `OPT3`, `FD3`, and `FD5` propagations. The comparison uses Gaussian initial fields rather than sources, so the benchmark tests propagation operators first. Cases: acoustic/elastic and homogeneous/Marmousi.

## 1. Setup

In [ ]:
# Locate flexOPT securely without relying on @__DIR__ (unreliable in IJulia).
# If this notebook is outside the repository, set ENV["FLEXOPT_ROOT"] first.
import Pkg

function find_flexopt_root(start_dir=pwd())
    candidates = String[]
    if haskey(ENV, "FLEXOPT_ROOT")
        push!(candidates, abspath(expanduser(ENV["FLEXOPT_ROOT"])))
    end
    directory = abspath(start_dir)
    while true
        push!(candidates, directory)
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end
    for candidate in unique(candidates)
        project_file = joinpath(candidate, "Project.toml")
        source_dir = joinpath(candidate, "src")
        if isfile(project_file) && isfile(joinpath(source_dir, "commonBatchs.jl"))
            return candidate
        end
    end
    error("Cannot locate flexOPT. Start Jupyter inside the repository or set ENV[\"FLEXOPT_ROOT\"] to its absolute path.")
end

flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)
@show VERSION Threads.nthreads() Base.active_project()
\n# Metal must be loaded before batchGPU.jl selects the backend.
using Metal
Metal.functional() || error("Metal.jl is loaded, but cannot access the Apple GPU")
@show Metal.devices()

using LinearAlgebra
using Statistics
using SparseArrays
using JLD2
using CairoMakie
CairoMakie.activate!()

include(joinpath(flexopt_root, "src", "batchFiles", "batchGPU.jl"))
include(joinpath(flexopt_root, "src", "commonBatchs.jl"))
include(joinpath(flexopt_root, "src", "planet1D.jl"))
planet1D.configure_input!()
include(joinpath(flexopt_root, "src", "GeoPoints.jl"))
using .commonBatchs, .planet1D, .GeoPoints

include(joinpath(flexopt_root, "src", "flexOPT.jl"))
using .flexOPT

include("temporaryHelpers.jl")\n

## 2. Benchmark Helpers

`refinement = 2` means the reference grid has twice as many points per direction and half the grid spacing. Reference frames are downsampled back to the coarse grid before comparison.

In [ ]:
function run_scalar_initial(prepared, initial2d, Nt; store_every=4, blowup_limit=1e12)
    frames_full = propagate_linear_frames_with_source(
        prepared,
        Nt;
        initialPast=initial2d,
        initialPresent=initial2d,
        store_every=store_every,
        blowup_limit=blowup_limit,
    )
    return component_frames(frames_full, 1)
end

function run_vector_initial(prepared, initial3d, Nt; store_every=4, blowup_limit=1e12)
    frames_full = propagate_linear_frames_with_source(
        prepared,
        Nt;
        initialPast=initial3d,
        initialPresent=initial3d,
        store_every=store_every,
        blowup_limit=blowup_limit,
    )
    return frames_full, component_frames(frames_full, 1), component_frames(frames_full, 2)
end

function acoustic_benchmark(label, velocity_ref, dx_ref; refinement=2, Nt=80, store_every=4, cfl=0.35, sigma_cells=8.0, run_opt=true)
    shape_ref = size(velocity_ref)
    velocity = downsample_frame_by(velocity_ref, refinement)
    shape = size(velocity)
    sourcePoint = CartesianIndex(cld(shape[1], 2), cld(shape[2], 2))
    sourcePoint_ref = CartesianIndex(1 + (sourcePoint[1] - 1) * refinement, 1 + (sourcePoint[2] - 1) * refinement)

    dx = dx_ref * refinement
    vmax = max(maximum(velocity), maximum(velocity_ref))
    dt = cfl * dx / (sqrt(2) * vmax)
    delta = (dx, dx, dt)
    delta_ref = (dx_ref, dx_ref, dt / refinement)
    Nt_ref = Nt * refinement
    store_ref = store_every * refinement

    initial = gaussian_field(shape, sourcePoint; sigma=sigma_cells, amplitude=1.0)
    initial_ref = gaussian_field(shape_ref, sourcePoint_ref; sigma=sigma_cells * refinement, amplitude=1.0)

    prepared_ref = prepare_fd2d_acoustic_fd5_baseline(velocity_ref, delta_ref)
    t_ref = @elapsed frames_ref_full = run_scalar_initial(prepared_ref, initial_ref, Nt_ref; store_every=store_ref)
    frames_ref = downsample_frames_by(frames_ref_full, refinement)

    prepared_fd3 = prepare_fd2d_acoustic_baseline(velocity, delta)
    t_fd3 = @elapsed frames_fd3 = run_scalar_initial(prepared_fd3, initial, Nt; store_every=store_every)

    prepared_fd5 = prepare_fd2d_acoustic_fd5_baseline(velocity, delta)
    t_fd5 = @elapsed frames_fd5 = run_scalar_initial(prepared_fd5, initial, Nt; store_every=store_every)

    opt3 = nothing
    prepared_opt3 = nothing
    frames_opt3 = nothing
    t_opt3 = NaN
    opt_report = nothing
    if run_opt
        t_opt3 = @elapsed begin
            opt3 = build_opt_prepared(
                "2DacousticTime",
                [velocity],
                delta;
                pointsInSpace=3,
                pointsInTime=3,
                supplementaryOrder=2,
                orderBspace=1,
                orderBtime=1,
                YorderBspace=-1,
                YorderBtime=-1,
                modelName=string(label, "_acoustic_OPT3"),
            )
            prepared_opt3 = opt3.prepared
            frames_opt3 = run_scalar_initial(prepared_opt3, initial, Nt; store_every=store_every)
        end
        opt_report = implicit_matrix_report(prepared_opt3)
    end

    reports = (
        fd3 = benchmark_report_against_reference(frames_fd3, frames_ref; label=:fd3),
        fd5 = benchmark_report_against_reference(frames_fd5, frames_ref; label=:fd5),
        opt3 = frames_opt3 === nothing ? nothing : benchmark_report_against_reference(frames_opt3, frames_ref; label=:opt3),
    )

    return (; label, shape, shape_ref, dx, dx_ref, dt, Nt, Nt_ref, store_every, refinement, sourcePoint,
        prepared_ref, prepared_fd3, prepared_fd5, prepared_opt3, opt3,
        frames_ref, frames_fd3, frames_fd5, frames_opt3,
        timings=(ref=t_ref, fd3=t_fd3, fd5=t_fd5, opt3=t_opt3),
        reports, opt_report)
end

function elastic_benchmark(label, rho_ref, lambda_ref, mu_ref, dx_ref; refinement=2, Nt=60, store_every=3, cfl=0.22, sigma_cells=8.0, run_opt=true)
    rho = downsample_frame_by(rho_ref, refinement)
    lambda = downsample_frame_by(lambda_ref, refinement)
    mu = downsample_frame_by(mu_ref, refinement)
    shape = size(rho)
    shape_ref = size(rho_ref)
    sourcePoint = CartesianIndex(cld(shape[1], 2), cld(shape[2], 2))
    sourcePoint_ref = CartesianIndex(1 + (sourcePoint[1] - 1) * refinement, 1 + (sourcePoint[2] - 1) * refinement)

    vp_ref = sqrt.((lambda_ref .+ 2 .* mu_ref) ./ rho_ref)
    vp = sqrt.((lambda .+ 2 .* mu) ./ rho)
    dx = dx_ref * refinement
    vmax = max(maximum(vp), maximum(vp_ref))
    dt = cfl * dx / (sqrt(2) * vmax)
    delta = (dx, dx, dt)
    delta_ref = (dx_ref, dx_ref, dt / refinement)
    Nt_ref = Nt * refinement
    store_ref = store_every * refinement

    initial = zeros(Float64, shape..., 2)
    initial[:, :, 1] .= gaussian_field(shape, sourcePoint; sigma=sigma_cells, amplitude=1.0)
    initial_ref = zeros(Float64, shape_ref..., 2)
    initial_ref[:, :, 1] .= gaussian_field(shape_ref, sourcePoint_ref; sigma=sigma_cells * refinement, amplitude=1.0)

    prepared_ref = prepare_fd2d_elastic_pointwise_baseline(rho_ref, lambda_ref, mu_ref, delta_ref; spatial_order=5)
    t_ref = @elapsed begin
        frames_ref_full, ux_ref_full, uz_ref_full = run_vector_initial(prepared_ref, initial_ref, Nt_ref; store_every=store_ref)
    end
    ux_ref = downsample_frames_by(ux_ref_full, refinement)
    uz_ref = downsample_frames_by(uz_ref_full, refinement)

    prepared_fd3 = prepare_fd2d_elastic_pointwise_baseline(rho, lambda, mu, delta; spatial_order=3)
    t_fd3 = @elapsed begin
        frames_fd3_full, ux_fd3, uz_fd3 = run_vector_initial(prepared_fd3, initial, Nt; store_every=store_every)
    end

    prepared_fd5 = prepare_fd2d_elastic_pointwise_baseline(rho, lambda, mu, delta; spatial_order=5)
    t_fd5 = @elapsed begin
        frames_fd5_full, ux_fd5, uz_fd5 = run_vector_initial(prepared_fd5, initial, Nt; store_every=store_every)
    end

    opt3 = nothing
    prepared_opt3 = nothing
    ux_opt3 = nothing
    uz_opt3 = nothing
    t_opt3 = NaN
    opt_report = nothing
    if run_opt
        t_opt3 = @elapsed begin
            opt3 = build_opt_prepared(
                "2DsismoTimeIsoHetero",
                [rho, lambda, mu],
                delta;
                pointsInSpace=3,
                pointsInTime=3,
                supplementaryOrder=2,
                orderBspace=1,
                orderBtime=1,
                YorderBspace=-1,
                YorderBtime=-1,
                modelName=string(label, "_elastic_OPT3"),
            )
            prepared_opt3 = opt3.prepared
            _, ux_opt3, uz_opt3 = run_vector_initial(prepared_opt3, initial, Nt; store_every=store_every, blowup_limit=1e30)
        end
        opt_report = implicit_matrix_report(prepared_opt3)
    end

    reports = (
        ux_fd3 = benchmark_report_against_reference(ux_fd3, ux_ref; label=:ux_fd3),
        ux_fd5 = benchmark_report_against_reference(ux_fd5, ux_ref; label=:ux_fd5),
        ux_opt3 = ux_opt3 === nothing ? nothing : benchmark_report_against_reference(ux_opt3, ux_ref; label=:ux_opt3),
        uz_fd3 = benchmark_report_against_reference(uz_fd3, uz_ref; label=:uz_fd3),
        uz_fd5 = benchmark_report_against_reference(uz_fd5, uz_ref; label=:uz_fd5),
        uz_opt3 = uz_opt3 === nothing ? nothing : benchmark_report_against_reference(uz_opt3, uz_ref; label=:uz_opt3),
    )

    return (; label, shape, shape_ref, dx, dx_ref, dt, Nt, Nt_ref, store_every, refinement, sourcePoint,
        prepared_ref, prepared_fd3, prepared_fd5, prepared_opt3, opt3,
        ux_ref, uz_ref, ux_fd3, uz_fd3, ux_fd5, uz_fd5, ux_opt3, uz_opt3,
        timings=(ref=t_ref, fd3=t_fd3, fd5=t_fd5, opt3=t_opt3),
        reports, opt_report)
end

function report_last_rows(result)
    out = NamedTuple[]
    for name in propertynames(result.reports)
        r = getproperty(result.reports, name)
        r === nothing && continue
        push!(out, r[end])
    end
    return out
end


## 3. Homogeneous Acoustic

Reference: `FD5` on a `200x200` grid. Candidates: `FD3`, `FD5`, `OPT3` on `100x100`.

In [ ]:
refinement_homo_ac = 2
shape_ac_ref = (200, 200)
dx_ac_ref = 25.0
v0_ac = 2600.0
velocity_ac_ref = fill(v0_ac, shape_ac_ref)

bench_ac_homo = acoustic_benchmark(
    :homo,
    velocity_ac_ref,
    dx_ac_ref;
    refinement=refinement_homo_ac,
    Nt=80,
    store_every=4,
    cfl=0.35,
    sigma_cells=8.0,
    run_opt=true,
)

@show bench_ac_homo.shape bench_ac_homo.shape_ref bench_ac_homo.dx bench_ac_homo.dx_ref bench_ac_homo.dt
@show bench_ac_homo.timings
report_last_rows(bench_ac_homo)


In [ ]:
display(plot_wave_snapshots(bench_ac_homo.frames_ref; sourcePoint=bench_ac_homo.sourcePoint, title="homo acoustic reference FD5 fine→coarse"))
display(plot_wave_snapshots(bench_ac_homo.frames_fd3; sourcePoint=bench_ac_homo.sourcePoint, title="homo acoustic FD3 coarse"))
display(plot_wave_snapshots(bench_ac_homo.frames_fd5; sourcePoint=bench_ac_homo.sourcePoint, title="homo acoustic FD5 coarse"))
bench_ac_homo.frames_opt3 === nothing ? nothing : plot_wave_snapshots(bench_ac_homo.frames_opt3; sourcePoint=bench_ac_homo.sourcePoint, title="homo acoustic OPT3 coarse")


## 4. Homogeneous Elastic

Reference: pointwise elastic `FD5` on `200x200`. Candidates: pointwise elastic `FD3`, `FD5`, and `OPT3` on `100x100`.

In [ ]:
refinement_homo_el = 2
shape_el_ref = (200, 200)
dx_el_ref = 25.0
rho0 = 2500.0
vp0 = 3200.0
vs0 = 1800.0
rho_el_ref = fill(rho0, shape_el_ref)
mu_el_ref = fill(rho0 * vs0^2, shape_el_ref)
lambda_el_ref = fill(rho0 * vp0^2 - 2rho0 * vs0^2, shape_el_ref)

bench_el_homo = elastic_benchmark(
    :homo,
    rho_el_ref,
    lambda_el_ref,
    mu_el_ref,
    dx_el_ref;
    refinement=refinement_homo_el,
    Nt=60,
    store_every=3,
    cfl=0.22,
    sigma_cells=8.0,
    run_opt=true,
)

@show bench_el_homo.shape bench_el_homo.shape_ref bench_el_homo.dx bench_el_homo.dx_ref bench_el_homo.dt
@show bench_el_homo.timings
report_last_rows(bench_el_homo)


In [ ]:
display(plot_wave_snapshots(bench_el_homo.ux_ref; sourcePoint=bench_el_homo.sourcePoint, title="homo elastic ux reference FD5 fine→coarse"))
display(plot_wave_snapshots(bench_el_homo.ux_fd3; sourcePoint=bench_el_homo.sourcePoint, title="homo elastic ux FD3 coarse"))
display(plot_wave_snapshots(bench_el_homo.ux_fd5; sourcePoint=bench_el_homo.sourcePoint, title="homo elastic ux FD5 coarse"))
bench_el_homo.ux_opt3 === nothing ? nothing : plot_wave_snapshots(bench_el_homo.ux_opt3; sourcePoint=bench_el_homo.sourcePoint, title="homo elastic ux OPT3 coarse")


## 5. Marmousi Acoustic

This uses a reduced Marmousi crop. The reference grid is still the fine grid for the benchmark; candidates are every `refinement` point.

In [ ]:
marmousi = load(joinpath(@__DIR__, "tmp/seismicModelMarmousi.jld2"), "output")

refinement_marm_ac = 2
shape_marm_ac_ref = (200, 200)
dx_marm_ac_ref = 50.0
# Use a modest pre-downsample to keep construction fast; tune this tomorrow if needed.
velocity_marm_ac_ref = downsample_center_crop(marmousi.Vpv .* 1e3, shape_marm_ac_ref; step=2)

bench_ac_marm = acoustic_benchmark(
    :marmousi,
    velocity_marm_ac_ref,
    dx_marm_ac_ref;
    refinement=refinement_marm_ac,
    Nt=80,
    store_every=4,
    cfl=0.30,
    sigma_cells=8.0,
    run_opt=true,
)

@show extrema(velocity_marm_ac_ref) bench_ac_marm.shape bench_ac_marm.shape_ref bench_ac_marm.dt
@show bench_ac_marm.timings
report_last_rows(bench_ac_marm)


In [ ]:
display(plot_wave_snapshots(bench_ac_marm.frames_ref; sourcePoint=bench_ac_marm.sourcePoint, title="Marmousi acoustic reference FD5 fine→coarse"))
display(plot_wave_snapshots(bench_ac_marm.frames_fd3; sourcePoint=bench_ac_marm.sourcePoint, title="Marmousi acoustic FD3 coarse"))
display(plot_wave_snapshots(bench_ac_marm.frames_fd5; sourcePoint=bench_ac_marm.sourcePoint, title="Marmousi acoustic FD5 coarse"))
bench_ac_marm.frames_opt3 === nothing ? nothing : plot_wave_snapshots(bench_ac_marm.frames_opt3; sourcePoint=bench_ac_marm.sourcePoint, title="Marmousi acoustic OPT3 coarse")


## 6. Marmousi Elastic

The FD elastic reference/candidates here are pointwise-coefficient baselines, not a conservative staggered-grid elastic solver. They are useful for comparing against OPT behavior but not a final elastic production solver.

In [ ]:
refinement_marm_el = 2
shape_marm_el_ref = (100, 100)
dx_marm_el_ref = 100.0
rho_raw_ref = downsample_center_crop(marmousi.ρ, shape_marm_el_ref; step=4)
vp_raw_ref = downsample_center_crop(marmousi.Vpv, shape_marm_el_ref; step=4)
vs_raw_ref = downsample_center_crop(marmousi.Vsv, shape_marm_el_ref; step=4)

rho_marm_el_ref, lambda_marm_el_ref, mu_marm_el_ref, vp_marm_el_ref, vs_marm_el_ref = elastic_lame_from_rho_vp_vs(rho_raw_ref, vp_raw_ref, vs_raw_ref)
vs_floor = 500.0
vs_marm_el_ref = max.(vs_marm_el_ref, vs_floor)
mu_marm_el_ref = rho_marm_el_ref .* vs_marm_el_ref.^2
lambda_marm_el_ref = rho_marm_el_ref .* vp_marm_el_ref.^2 .- 2 .* mu_marm_el_ref

bench_el_marm = elastic_benchmark(
    :marmousi,
    rho_marm_el_ref,
    lambda_marm_el_ref,
    mu_marm_el_ref,
    dx_marm_el_ref;
    refinement=refinement_marm_el,
    Nt=40,
    store_every=2,
    cfl=0.18,
    sigma_cells=6.0,
    run_opt=true,
)

@show extrema(rho_marm_el_ref) extrema(vp_marm_el_ref) extrema(vs_marm_el_ref)
@show bench_el_marm.shape bench_el_marm.shape_ref bench_el_marm.dt bench_el_marm.timings
report_last_rows(bench_el_marm)


In [ ]:
display(plot_wave_snapshots(bench_el_marm.ux_ref; sourcePoint=bench_el_marm.sourcePoint, title="Marmousi elastic ux reference FD5 fine→coarse"))
display(plot_wave_snapshots(bench_el_marm.ux_fd3; sourcePoint=bench_el_marm.sourcePoint, title="Marmousi elastic ux FD3 coarse"))
display(plot_wave_snapshots(bench_el_marm.ux_fd5; sourcePoint=bench_el_marm.sourcePoint, title="Marmousi elastic ux FD5 coarse"))
bench_el_marm.ux_opt3 === nothing ? nothing : plot_wave_snapshots(bench_el_marm.ux_opt3; sourcePoint=bench_el_marm.sourcePoint, title="Marmousi elastic ux OPT3 coarse")


## 7. Quick Access To Last Reports

In [ ]:
# Re-run this after executing any subset above.
for name in (:bench_ac_homo, :bench_el_homo, :bench_ac_marm, :bench_el_marm)
    if isdefined(Main, name)
        result = getfield(Main, name)
        println("\n", name, " timings = ", result.timings)
        display(report_last_rows(result))
    end
end
